#WEB SCRAPING DATA ULASAN APLIKASI MOBILE JKN  BERBASIS ULASAN GOOGLE PLAY STORE

In [ ]:
!pip install google-play-scraper pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import time
from google_play_scraper import Sort, reviews

APP_ID = 'app.bpjs.mobile'
LANG = 'id'
COUNTRY = 'id'
TARGET_COUNT = 3000
TAHUN_AWAL = 2025
TAHUN_AKHIR = 2026

# Kata kunci kualitas layanan (berdasarkan dimensi E-ServQual)
# Efficiency, Fulfillment, System Availability, Privacy
KEYWORDS_LAYANAN = [
    # Efficiency & Fulfillment
    'mudah', 'cepat', 'lambat', 'lemot', 'efisien', 'praktis', 'ribet',
    'sulit', 'gampang', 'sesuai', 'fungsi', 'fitur', 'manfaat', 'berguna',
    'membantu', 'gagal', 'sukses', 'login', 'daftar', 'bayar', 'iuran',
    # System Availability
    'error', 'gangguan', 'server', 'down', 'sistem', 'bug', 'crash',
    'macet', 'loading', 'jaringan', 'koneksi', 'force close', 'keluar sendiri',
    # Privacy & General Service
    'aman', 'data', 'privasi', 'password', 'akun', 'bocor', 'rahasia',
    'layanan', 'pelayanan', 'respon', 'cs', 'customer', 'bantuan',
    'komplain', 'bpjs', 'jkn', 'kartu', 'faskes', 'aplikasi', 'update'
]

# ====== PROSES SCRAPING ======
valid_reviews = []
continuation_token = None
stop_scraping = False
batch_count = 0

print("=" * 60)
print("🚀 MEMULAI SCRAPING ULASAN MOBILE JKN")
print(f"📱 App ID    : {APP_ID}")
print(f"📅 Rentang   : {TAHUN_AWAL} - {TAHUN_AKHIR}")
print(f"🎯 Target    : {TARGET_COUNT} ulasan")
print("=" * 60)

for i in range(200):  # Maksimal iterasi untuk antisipasi
    if stop_scraping or len(valid_reviews) >= TARGET_COUNT:
        break

    try:
        result, continuation_token = reviews(
            APP_ID,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,  # Ambil dari yang terbaru
            count=200,         # 200 ulasan per batch
            continuation_token=continuation_token
        )
    except Exception as e:
        print(f"⚠️ Error pada batch {i+1}: {e}")
        time.sleep(5)
        continue

    if not result:
        print("ℹ️ Tidak ada data lagi dari Google Play Store.")
        break

    batch_count += 1

    for r in result:
        content = r.get('content', '') or ''
        date = r.get('at')
        score = r.get('score')

        if date is None or content is None:
            continue

        # ====== FILTER 1: TAHUN 2025 - 2026 ======
        if date.year < TAHUN_AWAL:
            stop_scraping = True
            print(f"⏹️ Sudah melewati tahun {TAHUN_AWAL}. Scraping dihentikan.")
            break
        if date.year > TAHUN_AKHIR:
            continue

        # ====== FILTER 2: BUKAN 1 KATA (HARUS KALIMAT) ======
        words = content.split()
        if len(words) <= 5:
            continue

        # ====== FILTER 3: HARUS MEMBAHAS KUALITAS LAYANAN ======
        content_lower = content.lower()
        if not any(keyword in content_lower for keyword in KEYWORDS_LAYANAN):
            continue

        # ====== FILTER 4: BUKAN SPAM/POLA SAMA ======
        if len(set(content_lower)) < 5:  # Abaikan ulasan karakter berulang
            continue

        # Jika lolos semua filter, simpan
        valid_reviews.append({
            'review_id': r.get('reviewId'),
            'user_name': r.get('userName'),
            'date': date,
            'score': score,
            'review': content,
            'thumbs_up_count': r.get('thumbsUp'),
            'app_version': r.get('reviewCreatedVersion')        })

        # Jika sudah mencapai target, berhenti
        if len(valid_reviews) >= TARGET_COUNT:
            break

    print(f"⏳ Batch {batch_count}: Terkumpul {len(valid_reviews)}/{TARGET_COUNT} ulasan valid...")
    time.sleep(2)  # Jeda 2 detik untuk menghindari pemblokiran IP

# ====== SIMPAN KE DATAFRAME & CSV ======
df = pd.DataFrame(valid_reviews).head(TARGET_COUNT)

# Urutkan berdasarkan tanggal terbaru
df = df.sort_values(by='date', ascending=False).reset_index(drop=True)

# Simpan ke CSV
nama_file = 'ulasan_mobile_jkn_2025_2026.csv'
df.to_csv(nama_file, index=False, encoding='utf-8')

print("\n" + "=" * 60)
print("✅ PROSES SCRAPING SELESAI!")
print(f"📊 Total ulasan valid terkumpul : {len(df)}")
print(f"💾 File tersimpan sebagai       : {nama_file}")
print("=" * 60)

# Tampilkan ringkasan data
print("\n📋 RINGKASAN DATA:")
print(f"   Rentang tanggal : {df['date'].min()} s/d {df['date'].max()}")
print(f"   Distribusi rating:")
print(df['score'].value_counts().sort_index())
print("\n🔍 5 Ulasan Teratas:")
display(df[['date', 'score', 'review']].head())

🚀 MEMULAI SCRAPING ULASAN MOBILE JKN
📱 App ID    : app.bpjs.mobile
📅 Rentang   : 2025 - 2026
🎯 Target    : 3000 ulasan
⏳ Batch 1: Terkumpul 79/3000 ulasan valid...
⏳ Batch 2: Terkumpul 166/3000 ulasan valid...
⏳ Batch 3: Terkumpul 251/3000 ulasan valid...
⏳ Batch 4: Terkumpul 328/3000 ulasan valid...
⏳ Batch 5: Terkumpul 422/3000 ulasan valid...
⏳ Batch 6: Terkumpul 515/3000 ulasan valid...
⏳ Batch 7: Terkumpul 600/3000 ulasan valid...
⏳ Batch 8: Terkumpul 680/3000 ulasan valid...
⏳ Batch 9: Terkumpul 754/3000 ulasan valid...
⏳ Batch 10: Terkumpul 849/3000 ulasan valid...
⏳ Batch 11: Terkumpul 942/3000 ulasan valid...
⏳ Batch 12: Terkumpul 1012/3000 ulasan valid...
⏳ Batch 13: Terkumpul 1092/3000 ulasan valid...
⏳ Batch 14: Terkumpul 1174/3000 ulasan valid...
⏳ Batch 15: Terkumpul 1249/3000 ulasan valid...
⏳ Batch 16: Terkumpul 1327/3000 ulasan valid...
⏳ Batch 17: Terkumpul 1433/3000 ulasan valid...
⏳ Batch 18: Terkumpul 1534/3000 ulasan valid...
⏳ Batch 19: Terkumpul 1611/3000 ulasan

,date,score,review
0,2026-06-05 14:03:48,1,"Daftar autodebet gagal mulu notifnya, padahal ..."
1,2026-06-05 13:40:55,1,perbaiki dong aplikasinya masa nomor telpon ko...
2,2026-06-05 12:23:17,5,aplikasi sampah. eror terus harus jaringan 10G...
3,2026-06-05 12:12:31,5,aplikasi bagus prosesnya cepat dan menunya len...
4,2026-06-05 11:47:42,5,sangat baik dan bermanfaat sangat lebih cepat ...
